# 04b. RII 공간 이질성 변수 추가
## 불투수면 비율 + 저지대 비율 → RII v2

**배경**

기존 RII의 `rain_exposure_score`는 서울 전체 강수량 CSV(단일 집계)를 사용해  
모든 행정동에서 동일한 값이 되어 행정동 간 **공간 변별력이 없었습니다.**

이를 보완하기 위해 **행정동별로 값이 다른** 두 공간 변수를 추가합니다.

| 추가 변수 | 데이터 | 의미 |
|-----------|--------|------|
| `impervious_ratio` | 도시생태현황도(비오톱) | 불투수면 비율 높을수록 빗물 침투 불가 → 침수 위험 ↑ |
| `low_elevation_score` | DEM(수치표고모델) | 저지대일수록 빗물 집수 → 침수 위험 ↑ |

**새 RII 구성 (5개 구성요소 가중 평균)**

| 구성요소 | 가중치 | 데이터 |
|----------|--------|--------|
| flood_trace_score | 0.30 | 침수흔적도 (과거 피해 이력) |
| flood_prediction_score | 0.25 | 홍수예측 (구조적 위험) |
| impervious_score | 0.25 | 불투수면 비율 (강우 시 침수 가속) |
| low_elevation_score | 0.15 | 저지대 비율 (물 집수 지형) |
| rain_exposure_score | 0.05 | 강수량 (참고용, 전동 동일) |

> rain_exposure_score는 전 행정동 동일값이므로 가중치를 최소화하되 완전 제거하지 않아  
> 기존 RII 계산 구조와의 연속성을 유지합니다.

---
## 1. 라이브러리 및 경로 설정

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from pathlib import Path
from shapely.geometry import mapping

BASE_DIR  = Path('c:/Tsum2026/T_SUM2026')
RII_RAW   = BASE_DIR / 'data' / 'raw' / 'RII_rain_flood'
PROC_DIR  = BASE_DIR / 'data' / 'processed'

# 입력 경로
ADM_SHP_PATH  = RII_RAW / 'map_seoul_dong_2021' / 'BND_ADM_DONG_PG.shp'
BIO_SHP_PATH  = RII_RAW / 'eco_biotope_2021' / 'UPIS_BIOTOP_SEP.shp'
DEM_PATH      = RII_RAW / 'topo_dem_seoul_2021.tif.img'
RII_OLD_PATH  = PROC_DIR / 'rii_by_dong_2021.csv'

# 출력 경로
OUT_RII_V2_CSV  = PROC_DIR / 'rii_by_dong_2021_v2.csv'
OUT_RII_V2_GPKG = PROC_DIR / 'rii_by_dong_2021_v2.gpkg'

TARGET_CRS = 'EPSG:5179'

for label, path in [
    ('행정동 경계 SHP', ADM_SHP_PATH),
    ('도시생태현황도 SHP', BIO_SHP_PATH),
    ('DEM 파일', DEM_PATH),
    ('기존 RII CSV', RII_OLD_PATH),
]:
    mark = '✓' if path.exists() else '✗ 없음!'
    print(f'[{mark}] {label}: {path}')

---
## 2. 행정동 경계 로드 및 코드 매핑

- `ADM_CD` (8자리: `11010530`) → 앞 7자리 = RII 행정동코드 (`1101053`)
- 서울(코드 11 시작) 425개 행정동만 필터링

In [ ]:
# 행정동 경계 로드 (전국 → 서울 필터링)
adm_all = gpd.read_file(str(ADM_SHP_PATH))
print(f'전국 행정동 수: {len(adm_all)}')

# 서울 필터링 (ADM_CD 앞 2자리 = '11')
adm = adm_all[adm_all['ADM_CD'].astype(str).str[:2] == '11'].copy()
adm = adm.reset_index(drop=True)
print(f'서울 행정동 수: {len(adm)} | 원본 CRS: {adm.crs}')

# 7자리 행정동코드 생성 (ADM_CD 앞 7자리)
adm['행정동코드'] = adm['ADM_CD'].astype(str).str[:7].astype(int)

# EPSG:5179로 변환
adm = adm.to_crs(TARGET_CRS)

# 행정동 면적 계산 (m²)
adm['dong_area_m2'] = adm.geometry.area

# RII 코드와 매핑 확인
rii_old = pd.read_csv(RII_OLD_PATH, encoding='utf-8-sig')
rii_codes = set(rii_old['행정동코드'].astype(int))
adm_codes = set(adm['행정동코드'])
print(f'\nRII 코드 수: {len(rii_codes)}, 행정동경계 코드 수: {len(adm_codes)}')
print(f'RII에만 있는 코드: {sorted(rii_codes - adm_codes)[:5]}')
print(f'경계에만 있는 코드 수: {len(adm_codes - rii_codes)}')

display(adm[['ADM_CD', '행정동코드', 'ADM_NM', 'dong_area_m2']].head())

---
## 3. 불투수면 비율 계산

**불투수면 정의**: 비오톱유형 접두사 `B`(건물·인공구조물) + `C`(교통·도로시설)  
→ 비가 와도 지면으로 침투하지 않아 지표 유출 발생 → 침수 위험 증가

In [ ]:
# 도시생태현황도 로드 (cp949 인코딩)
bio = gpd.read_file(str(BIO_SHP_PATH), encoding='cp949')
print(f'비오톱 폴리곤 수: {len(bio)}')
print(f'원본 CRS: {bio.crs}')

# CRS 없음 → bounds 기반으로 EPSG:5186 가정 (서울시 도시생태현황도 표준)
if bio.crs is None:
    bio = bio.set_crs('EPSG:5186', allow_override=True)
    print('[주의] CRS 없어 EPSG:5186으로 설정 (bounds 기반 추정)')

# EPSG:5179로 변환
bio = bio.to_crs(TARGET_CRS)
print(f'변환 후 CRS: {bio.crs}')

# 불투수면 필터링: 비오톱유형 첫 글자가 B 또는 C
bio['is_impervious'] = bio['비오톱유형'].str[0].isin(['B', 'C'])
impervious = bio[bio['is_impervious']].copy()
print(f'\n불투수면 폴리곤 수: {len(impervious)} (전체의 {len(impervious)/len(bio)*100:.1f}%)')
print('불투수면 비오톱유형 분포 (상위 10):')
print(impervious['비오톱유형'].value_counts().head(10).to_dict())

In [ ]:
# 행정동별 불투수면 면적 공간 결합 (overlay)
# 불투수면 폴리곤 × 행정동 경계 → 교차 면적 합산
print('공간 결합 중... (시간이 소요될 수 있습니다)')

adm_simple = adm[['행정동코드', 'ADM_NM', 'dong_area_m2', 'geometry']].copy()
impervious_simple = impervious[['비오톱유형', 'geometry']].copy()

# geometry 유효성 보정
adm_simple['geometry'] = adm_simple.geometry.buffer(0)
impervious_simple['geometry'] = impervious_simple.geometry.buffer(0)

# sjoin으로 각 불투수면 폴리곤이 어느 행정동에 속하는지 결합
bio_joined = gpd.sjoin(impervious_simple, adm_simple[['행정동코드', 'dong_area_m2', 'geometry']],
                       how='left', predicate='intersects')

print(f'결합 결과: {len(bio_joined)}행, 행정동 매칭: {bio_joined["행정동코드"].notna().sum()}건')

# 행정동별 불투수면 면적 합산 (geometry.area로 각 폴리곤 면적 사용)
bio_joined['impervious_area'] = bio_joined.geometry.area
impervious_by_dong = (
    bio_joined.dropna(subset=['행정동코드'])
    .groupby('행정동코드')['impervious_area']
    .sum()
    .reset_index()
    .rename(columns={'impervious_area': 'impervious_area_m2'})
)
print(f'\n행정동별 불투수면 집계 완료: {len(impervious_by_dong)}개 동')

# 행정동 면적과 병합 → 불투수면 비율
dong_impervious = adm_simple[['행정동코드', 'dong_area_m2']].merge(
    impervious_by_dong, on='행정동코드', how='left'
)
dong_impervious['impervious_area_m2'] = dong_impervious['impervious_area_m2'].fillna(0)
dong_impervious['impervious_ratio'] = (
    dong_impervious['impervious_area_m2'] / dong_impervious['dong_area_m2']
).clip(0, 1)

print('\n=== 불투수면 비율 분포 ===')
print(dong_impervious['impervious_ratio'].describe().round(4))

display(dong_impervious.sort_values('impervious_ratio', ascending=False).head(10))

---
## 4. 저지대 비율 계산 (DEM)

**저지대 기준**: 서울 전체 픽셀 하위 25% 표고(약 14m) 이하  
→ 한강변, 하천변, 분지형 저지대에 해당

In [ ]:
print('DEM 처리 중...')
elevation_stats = []

with rasterio.open(str(DEM_PATH)) as src:
    print(f'DEM CRS: {src.crs} | 해상도: {src.res}m | Shape: {src.height}×{src.width}')
    
    # 전체 DEM에서 저지대 기준선 계산
    dem_all = src.read(1)
    nodata = src.nodata if src.nodata else -9999
    valid_mask = dem_all != nodata
    valid_elev = dem_all[valid_mask]
    lowland_threshold = float(np.percentile(valid_elev, 25))  # 하위 25% 표고
    print(f'저지대 기준 표고 (하위 25%): {lowland_threshold:.1f}m')
    print(f'서울 전체 표고 범위: {valid_elev.min():.1f}~{valid_elev.max():.1f}m, 평균: {valid_elev.mean():.1f}m')
    
    # 행정동 경계를 DEM CRS로 변환 (EPSG:5179 동일)
    adm_for_dem = adm_simple.to_crs(src.crs)
    
    for idx, row in adm_for_dem.iterrows():
        try:
            geom = [mapping(row.geometry)]
            out_image, _ = rio_mask(src, geom, crop=True, nodata=nodata)
            elev = out_image[0]
            valid = elev[elev != nodata]
            valid = valid[valid > -100]
            if len(valid) == 0:
                elevation_stats.append({
                    '행정동코드': row['행정동코드'],
                    'mean_elevation': np.nan,
                    'lowland_pixel_ratio': np.nan,
                })
                continue
            lowland_pixels = int((valid <= lowland_threshold).sum())
            elevation_stats.append({
                '행정동코드': row['행정동코드'],
                'mean_elevation': float(valid.mean()),
                'lowland_pixel_ratio': lowland_pixels / len(valid),
            })
        except Exception as e:
            elevation_stats.append({
                '행정동코드': row['행정동코드'],
                'mean_elevation': np.nan,
                'lowland_pixel_ratio': np.nan,
            })

df_elev = pd.DataFrame(elevation_stats)

# 결측 처리 (경계 밖 행정동 등)
miss_elev = df_elev['mean_elevation'].isna().sum()
if miss_elev > 0:
    print(f'\n표고 결측 {miss_elev}건 → 서울 평균으로 대체')
    df_elev['mean_elevation'].fillna(valid_elev.mean(), inplace=True)
    df_elev['lowland_pixel_ratio'].fillna(0.25, inplace=True)  # 전체 평균 비율

print(f'\n=== 행정동별 표고 통계 ===')
print(df_elev[['mean_elevation', 'lowland_pixel_ratio']].describe().round(4))

print('\n저지대 비율 상위 10개 (침수 위험 높은 지형):')
top_low = df_elev.sort_values('lowland_pixel_ratio', ascending=False).head(10)
display(top_low.merge(adm[['행정동코드','ADM_NM']], on='행정동코드').to_string(index=False))

---
## 5. 공간 변수 정규화 및 새 RII 산출

In [ ]:
def minmax_scale(series):
    s = pd.to_numeric(series, errors='coerce').fillna(0)
    if s.max() == s.min():
        return pd.Series(0.5, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

# 기존 RII 로드
rii = pd.read_csv(RII_OLD_PATH, encoding='utf-8-sig')
rii['행정동코드'] = rii['행정동코드'].astype(int)

# 불투수면 비율 병합
rii = rii.merge(
    dong_impervious[['행정동코드', 'impervious_ratio']],
    on='행정동코드', how='left'
)

# 저지대 비율 병합
rii = rii.merge(df_elev[['행정동코드','mean_elevation','lowland_pixel_ratio']],
                on='행정동코드', how='left')

# 결측 확인
print('불투수면 비율 결측:', rii['impervious_ratio'].isna().sum())
print('저지대 비율 결측:', rii['lowland_pixel_ratio'].isna().sum())
rii['impervious_ratio'] = rii['impervious_ratio'].fillna(rii['impervious_ratio'].median())
rii['lowland_pixel_ratio'] = rii['lowland_pixel_ratio'].fillna(rii['lowland_pixel_ratio'].median())

# 공간 변수 정규화
# 불투수면: 높을수록 위험 → 그대로 Min-Max
rii['impervious_score'] = minmax_scale(rii['impervious_ratio'])
# 저지대: 저지대 비율 높을수록 위험 → 그대로 Min-Max
rii['low_elevation_score'] = minmax_scale(rii['lowland_pixel_ratio'])

print('\n=== 신규 공간 변수 분포 ===')
for col in ['impervious_score', 'low_elevation_score']:
    s = rii[col]
    print(f'  [{col}] mean={s.mean():.3f} std={s.std():.3f} min={s.min():.3f} max={s.max():.3f}')

# 기존 구성요소가 있는지 확인
existing_scores = [c for c in ['rain_exposure_score','flood_trace_score','flood_prediction_score']
                   if c in rii.columns]
print(f'\n기존 RII 구성요소: {existing_scores}')

In [ ]:
# 새 RII 가중 평균 산출
# 기존 구성요소가 있으면 그대로 사용, 없으면 기존 RII에서 역산
NEW_WEIGHTS = {
    'flood_trace_score':      0.30,
    'flood_prediction_score': 0.25,
    'impervious_score':       0.25,
    'low_elevation_score':    0.15,
    'rain_exposure_score':    0.05,
}

# 기존 구성요소 없는 경우 RII 원값을 flood_trace+flood_prediction 합산 대체로 사용
for col in ['rain_exposure_score', 'flood_trace_score', 'flood_prediction_score']:
    if col not in rii.columns:
        # 기존 RII를 해당 구성요소 자리에 근사 대입
        rii[col] = rii['RII'].copy()
        print(f'[주의] {col} 없어 기존 RII 값으로 대체')

rii['RII_v2'] = sum(
    rii[col].fillna(0) * w for col, w in NEW_WEIGHTS.items()
)

# 순위·등급 재계산
rii['RII_v2_rank'] = rii['RII_v2'].rank(ascending=False, method='min').astype(int)
rii['RII_v2_grade'] = pd.qcut(
    rii['RII_v2'].rank(method='first'), q=5, labels=[1,2,3,4,5], duplicates='drop'
).astype(int)

print('=== 기존 RII vs 신규 RII_v2 비교 ===')
print(f'기존 RII: mean={rii["RII"].mean():.3f} std={rii["RII"].std():.3f}')
print(f'신규 RII_v2: mean={rii["RII_v2"].mean():.3f} std={rii["RII_v2"].std():.3f}')
print(f'두 지수 상관계수: {rii["RII"].corr(rii["RII_v2"]):.3f}')
print()
print('=== RII_v2 상위 15개 (침수·지형 위험 복합) ===')
display(
    rii.nsmallest(15, 'RII_v2_rank')[
        ['자치구명','행정동명','RII','RII_v2',
         'flood_trace_score','flood_prediction_score',
         'impervious_score','low_elevation_score','RII_v2_rank']
    ].round(3)
)

---
## 6. 최종 저장

In [ ]:
# 최종 RII v2 컬럼 구성
# 기존 RII를 RII로 그대로 유지하고, RII_v2를 새 지수로 저장
# 이후 CCI에서 RII_v2를 RII로 사용
rii_out = rii.copy()
rii_out['RII'] = rii_out['RII_v2']           # CCI 재산출 시 RII 컬럼 덮어쓰기
rii_out['RII_rank'] = rii_out['RII_v2_rank']
rii_out['RII_grade'] = rii_out['RII_v2_grade']

rii_out.to_csv(OUT_RII_V2_CSV, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUT_RII_V2_CSV}')

# GPKG 저장 (지도용)
adm_geo = adm[['행정동코드','geometry']].copy()
rii_geo = adm_geo.merge(
    rii_out[['행정동코드','자치구명','행정동명',
             'impervious_ratio','impervious_score',
             'mean_elevation','lowland_pixel_ratio','low_elevation_score',
             'RII','RII_rank','RII_grade']],
    on='행정동코드', how='left'
)
rii_geo = gpd.GeoDataFrame(rii_geo, geometry='geometry', crs=adm.crs)
rii_geo.to_file(str(OUT_RII_V2_GPKG), layer='rii_v2_2021', driver='GPKG')
print(f'저장 완료: {OUT_RII_V2_GPKG}')

# 검증
print(f'\n=== 최종 검증 ===')
v = pd.read_csv(OUT_RII_V2_CSV, encoding='utf-8-sig')
print(f'행 수: {len(v)}')
print(f'RII 결측: {v["RII"].isna().sum()}')
print(f'RII 0~1: {bool(v["RII"].between(0,1).all())}')
print(f'impervious_score 변별력 std: {v["impervious_score"].std():.3f}')
print(f'low_elevation_score 변별력 std: {v["low_elevation_score"].std():.3f}')
print('✓ 검증 완료')